# Notebook for testing integration of processes together

In [2]:
from calculator.pricing import blended_price_per_1m, api_monthly_cost_usd, FLASH_PRICES
from calculator.benchmarks import self_host_config_for
from calculator.decision import evaluate_workload
from calculator.models import WorkloadInputs

In [3]:
## End-to-end scenario grid

scenarios = [
    ("Low volume should be API", 1_000_000),
    ("Mid volume likely API",    10_000_000),
    ("Higher volume maybe self-host depending on OH", 200_000_000),
]

for name, tpd in scenarios:
    w = WorkloadInputs(
        total_tokens_per_day=tpd,
        alpha_in=0.50, beta_out=0.50, gamma_think=0.00,   # "simple" split
        model_class="small",
        utilisation_target=0.75,
        overhead_rate=0.30,
        context_bucket="<=200k",
    )
    r = evaluate_workload(w)
    print(name)
    print("  tokens/day:", tpd)
    print("  api/mo:", round(r.api_monthly_usd, 2))
    print("  self/mo:", round(r.self_host_monthly_usd, 2))
    print("  rec:", r.recommendation)
    print("  replicas:", r.replicas_required)
    print()

Low volume should be API
  tokens/day: 1000000
  api/mo: 11.25
  self/mo: 411.84
  rec: API
  replicas: 1

Mid volume likely API
  tokens/day: 10000000
  api/mo: 112.5
  self/mo: 411.84
  rec: API
  replicas: 1

Higher volume maybe self-host depending on OH
  tokens/day: 200000000
  api/mo: 2250.0
  self/mo: 411.84
  rec: SELF_HOST
  replicas: 1



In [4]:
## Deterministic monotonicity checks

w1 = WorkloadInputs(10_000_000, 0.5,0.5,0.0, "small", 0.75, 0.30, "<=200k")
w2 = WorkloadInputs(20_000_000, 0.5,0.5,0.0, "small", 0.75, 0.30, "<=200k")

r1 = evaluate_workload(w1)
r2 = evaluate_workload(w2)

assert r2.api_monthly_usd > r1.api_monthly_usd
assert abs(r2.api_monthly_usd - 2*r1.api_monthly_usd) < 1e-6  # linear scaling
print("API monotonicity test passed.")

API monotonicity test passed.


In [5]:
## Replica step behaviour (self-host should nott decrease)

w_small = WorkloadInputs(50_000_000, 0.4,0.3,0.3, "large", 0.75, 0.30, "<=200k")
w_big   = WorkloadInputs(500_000_000,0.4,0.3,0.3, "large", 0.75, 0.30, "<=200k")

r_small = evaluate_workload(w_small)
r_big   = evaluate_workload(w_big)

assert r_big.replicas_required >= r_small.replicas_required
assert r_big.self_host_monthly_usd >= r_small.self_host_monthly_usd
print("Replica step test passed.")

Replica step test passed.
